# D5 LLM 활용 — 실습 (W12)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> (distilgpt2 모델은 첫 실행 시 자동 다운로드됩니다 — 약 350MB. 이 수업 PC에는 캐시돼 있습니다.)

**이 실습이 끝나면**
1. 토큰화 실측 — 서브워드·`Ġ`, 그리고 **한국어 24토큰 vs 영어 4토큰**(비용 감각)
2. **다음 토큰 확률 예측 + greedy 자기회귀**를 미니 LLM(distilgpt2)으로 확인한다
3. **온도(temperature) 손계산**([0.67, 0.24, 0.09] → T=2 [0.51, 0.31, 0.19])을 검증하고 실제 분포에 적용한다
4. distilgpt2 파라미터 **81,912,576개**를 세어 규모 감각을 잡는다 (D1c numel)
5. **API 헬퍼**(Claude·GPT 제공사 무관)로 분류·요약·형식 지정 — 프롬프트만 교체

**7단계 멘탈모델 초점:** 활용 (개념 + API)

## Part A. 토큰화 — LLM의 화폐 단위
LLM은 글자/단어가 아니라 **토큰**(서브워드 조각) 단위로 입출력합니다. D3b의 단어 사전 5,000+`<unk>`와 달리, 조각 사전(50,257개)에는 `<unk>`가 없습니다.

In [ ]:
import warnings; warnings.filterwarnings('ignore')   # 출력 깔끔하게
from transformers import AutoTokenizer                # 토크나이저
tok = AutoTokenizer.from_pretrained('distilgpt2')     # 미니 LLM용(첫 실행 시 다운로드)

text = 'Machine learning is fun'                      # 영어 예문
ids = tok(text)['input_ids']                          # 토큰 id 시퀀스
print('토큰:', tok.convert_ids_to_tokens(ids))        # 사람이 보는 토큰(Ġ=공백 표시)
print('id  :', ids)                                   # 모델이 받는 정수
print('쪼개짐:', tok.convert_ids_to_tokens(tok('tokenization')['input_ids']))  # 드문 단어는 서브워드로

In [ ]:
kor = '머신러닝은 재미있다'                           # 같은 뜻의 한국어(10글자)
n_kor = len(tok(kor)['___'])                          # ✍️ 빈칸: 토큰 id 리스트의 키 이름
print('영어  4단어 →', len(ids), '토큰')              # 흔한 영어 단어 = 1토큰
print('한국어 10글자 →', n_kor, '토큰')               # 잘게 쪼개짐!

> **4 vs 24** — 영어 위주로 학습된 토크나이저에게 한국어는 "드문 문자열"이라 잘게 쪼개집니다. API 과금은 **입력+출력 토큰 수** 기준이므로, 같은 뜻이라도 한국어 프롬프트는 **비용이 몇 배**가 됩니다(토큰 = 화폐 단위).

## Part B. 다음 토큰 예측 — "생성"의 전부
지금까지의 토큰 → **다음 토큰의 확률 분포**(어휘 전체 softmax — D4a의 그것) → 하나 골라 붙이고 반복.

In [ ]:
import torch                                          # 텐서
from transformers import AutoModelForCausalLM         # 다음 토큰(디코더형) 모델
model = AutoModelForCausalLM.from_pretrained('distilgpt2'); model.eval()  # 로드+평가 모드(D1c)

prompt = 'The capital of France is'                   # 시작 문장
inp = tok(prompt, return_tensors='pt')                # 토큰화
with torch.no_grad():                                 # 추론만(D1c no_grad)
    logits = model(**inp).logits                      # (1, 길이, 어휘 50257) 점수
next_logits = logits[0, -1]                           # 마지막 위치 = 다음 토큰 점수
probs = torch.softmax(next_logits, dim=___)           # ✍️ 빈칸: 어휘 전체(마지막 차원)로 확률화
top = torch.topk(probs, 5)                            # 확률 상위 5개
print('다음 토큰 후보(확률):')
for p, i in zip(top.values, top.indices):
    print(f'  {tok.decode(i)!r}: {p.item():.3f}')

In [ ]:
cur = inp['input_ids']                                # 자기회귀로 6토큰 생성(greedy)
for _ in range(6):
    with torch.no_grad():
        lg = model(cur).logits[0, -1]                 # ① 다음 토큰 점수
    nxt = torch.___(torch.softmax(lg, dim=-1))        # ✍️ 빈칸: ② 가장 확률 높은 토큰 = greedy
    cur = torch.cat([cur, nxt.view(1, 1)], dim=___)   # ✍️ 빈칸: ③ 시간(토큰) 차원 뒤에 붙여 반복
print('continuation:', tok.decode(cur[0]))            # 생성 결과

> **'Paris'가 상위에 없고**, 이어 붙이면 *"the capital of the French Republic"* — 같은 말을 맴돕니다. 8,200만짜리 미니 LLM의 정직한 민낯. 하지만 **루프·softmax·온도의 원리는 상용 LLM과 완전히 동일** — 차이는 규모뿐입니다.

## Part C. 온도의 정체 — 손계산 검증 ⭐
온도 = **softmax에 넣기 전에 점수를 T로 나누는 것.** 어휘 3개(Paris 2 · London 1 · pizza 0)로 손계산 후 검증합니다.
**먼저 종이에서** T=1과 T=2를 완주해 보세요 (e²=7.39, e¹=2.72, e⁰·⁵=1.65).

In [ ]:
mini = torch.tensor([2.0, 1.0, 0.0])                  # [Paris, London, pizza] 점수(logit)
p1 = torch.softmax(mini / 1.0, dim=0)                 # T=1: 기본
p2 = torch.softmax(mini / ___, dim=0)                 # ✍️ 빈칸: 높은 온도 T=2 (점수가 절반으로)
p05 = torch.softmax(mini / 0.5, dim=0)                # T=0.5: 낮은 온도(점수가 2배로)
for name, p in [('T=1  ', p1), ('T=2  ', p2), ('T=0.5', p05)]:
    print(name, [round(v, 2) for v in p.tolist()])    # 손계산과 대조

> **검산 포인트:** T=1 → [0.67, 0.24, 0.09] (e값 7.39/2.72/1.00, 합 11.11 — **D4a orange 행 [0.09, 0.67, 0.24]와 같은 숫자들!**), T=2 → [0.51, 0.31, 0.19] (평평 — pizza 확률 T=0.5의 2% → 19%), T=0.5 → [0.87, 0.12, 0.02] (쏠림). D4a에서 √dₖ로 나눠 쏠림을 *막았다면*, 온도는 T로 나눠 쏠림을 *조절*합니다.

In [ ]:
import matplotlib.pyplot as plt                       # 그래프
idxs = torch.topk(torch.softmax(next_logits, dim=-1), 5).indices  # T=1 기준 상위 5 토큰
fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.0), sharey=True)
for ax, T in zip(axes, [0.5, 1.0, 2.0]):              # 실제 분포에 온도 적용
    pT = torch.softmax(next_logits / ___, dim=-1)     # ✍️ 빈칸: 점수를 무엇으로 나누나?
    ax.bar(range(5), [pT[i].item() for i in idxs])    # 상위 5 후보의 확률
    ax.set_xticks(range(5))
    ax.set_xticklabels([repr(tok.decode(i)) for i in idxs], rotation=30, fontsize=8)
    ax.set_title(f'T = {T}')                          # 제목(영어)
    ax.set_ylim(0, 0.5)
axes[0].set_ylabel('probability')                     # 축(영어)
fig.suptitle('Next-token distribution vs temperature (distilgpt2)', fontsize=11)
fig.tight_layout(); plt.show()

> 실제 50,257개 분포에서도 같은 일이 벌어집니다: ' the'의 확률이 **T=0.5 → 0.459 / T=1 → 0.118 / T=2 → 0.007**. 낮은 T는 쏠림(안정·반복), 높은 T는 평평(다양·엉뚱). 같은 프롬프트에 답이 매번 다른 이유(비결정성)가 바로 T>0의 샘플링입니다.

## Part D. 규모 감각 — 파라미터 세기 (D1c numel)

In [ ]:
n = sum(p.___() for p in model.parameters())          # ✍️ 빈칸: 각 텐서의 원소 수(D1c의 그 함수)
print('distilgpt2 파라미터:', f'{n:,}')               # 81,912,576
print('블록 수:', model.config.n_layer, '| d_model:', model.config.n_embd)  # 6층, 768
print('어휘:', tok.vocab_size)                        # 50,257 조각

> **6층 × d_model 768** — D4b DistilBERT(66,985,530)와 같은 골격의 **디코더형 쌍둥이**입니다(차이는 주로 어휘 크기: 50,257 vs 30,522). 이 8,200만도 "France→Paris"가 벅찬데, GPT-3(2020)는 1,750억 — **약 2,100배**. 규모가 상식·추론을 창발시키는 대신, 내 PC엔 안 들어갑니다 → **API로 부립니다.**

## Part E. 상용 LLM API — 프롬프트가 스위치다
사전학습 LLM을 **추가 학습 없이 프롬프트(지시문+입력)만 바꿔** 씁니다.

> ⚠️ **API 키가 필요합니다(소액 과금).** 제공사 콘솔에서 발급 후 환경변수로 설정하세요.
> - Colab 예: ```python
> import os; from getpass import getpass
> os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic key: ')   # 또는 OPENAI_API_KEY
> ```
> - 라이브러리: `!pip install anthropic openai -q`
> - **키가 없으면** 아래 셀은 안내 문구만 출력하고 에러 없이 넘어갑니다(원리 학습엔 지장 없음).

In [ ]:
import os
# 제공사 무관 헬퍼: provider만 바꾸면 Claude/GPT 둘 다 호출(키 있을 때만 실제 호출)
def ask_llm(prompt, provider='anthropic', model=None, max_tokens=200):
    if provider == 'anthropic':
        key = os.environ.get('ANTHROPIC_API_KEY')
        if not key:
            return '[ANTHROPIC_API_KEY 미설정 — 키 설정 후 실행하면 실제 응답]'
        from anthropic import Anthropic                          # 키 있을 때만 import
        client = Anthropic(api_key=key)
        msg = client.messages.create(model=model or 'claude-haiku-4-5',  # 저렴한 모델
                                     max_tokens=max_tokens,
                                     messages=[{'role': 'user', 'content': prompt}])
        return msg.content[0].text
    else:  # openai
        key = os.environ.get('OPENAI_API_KEY')
        if not key:
            return '[OPENAI_API_KEY 미설정 — 키 설정 후 실행하면 실제 응답]'
        from openai import OpenAI
        client = OpenAI(api_key=key)
        resp = client.chat.completions.create(model=model or 'gpt-4o-mini',
                                              max_tokens=max_tokens,
                                              messages=[{'role': 'user', 'content': prompt}])
        return resp.choices[0].message.content

print('헬퍼 준비 완료. provider="anthropic"(Claude) 또는 "openai"(GPT)')

### 작업 1) 감정 분류 — 명확한 지시("하나로만")

In [ ]:
reviews = ['배송도 빠르고 품질 최고예요!',            # 분류할 리뷰들
           '한 달 만에 고장났고 환불도 안 됨.']
for r in reviews:
    prompt = '다음 리뷰를 긍정/부정 중 하나로만 분류해. 리뷰: "' + r + '"'  # 지시문+입력
    print(r, '→', ask_llm(prompt, provider='anthropic'))   # 키 없으면 안내 문구

### 작업 2) 요약, 그리고 형식 지정 — **지시문만** 바꾸면 끝

In [ ]:
article = ('인공지능 기술이 빠르게 발전하면서 의료, 교육, 제조 등 다양한 분야에 도입되고 있다. '
           '전문가들은 생산성 향상을 기대하면서도 일자리 변화와 윤리 문제에 대비해야 한다고 말한다.')
print('[요약]', ask_llm('다음 글을 한 문장으로 요약해 줘: ' + article, max_tokens=120))

fmt = '다음 리뷰의 감정을 JSON으로만 답해. 형식: {"sentiment": "긍정" 또는 "부정"} 리뷰: "그냥 평범해요"'
print('[형식 지정]', ask_llm(fmt, max_tokens=60))     # 출력 형식 지정 — 프로그램이 받기 좋게

> **같은 헬퍼·같은 모델, 지시문만 3번 교체**했습니다(분류→요약→형식 지정). 1학기 sklearn은 작업마다 새로 학습, D2c 전이학습은 마지막 층 재학습 — LLM은 **재학습 0**. 단, 응답은 확률 생성물이므로 **항상 검증**하세요(환각).

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "[2, 1, 0] → [0.67, 0.24, 0.09]를 내가 손으로 유도해 볼 테니 채점해 줘."
- "T=2에서 [0.51, 0.31, 0.19]가 되는 계산을 검산해 줘 (e⁰·⁵=1.65)."
- "한국어가 24토큰이 되는 이유와 비용 함의를 내 말로 설명해 볼게."
- "환각이 버그가 아니라 구조인 이유를 '다음 토큰 생성'으로 설명해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력(API 응답 포함)은 실행·사실 확인으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 토큰화 실측(한국어 24 vs 영어 4)으로 "토큰 = 화폐 단위" 감각을 잡았다
2. 미니 LLM으로 다음 토큰 확률·greedy 루프를 확인하고, **온도 손계산**([0.51, 0.31, 0.19])을 실제 분포(0.459/0.118/0.007)로 검증했다
3. 파라미터 81,912,576을 세고(규모=힘), API 헬퍼로 **프롬프트만 3번 교체**해 분류·요약·형식 지정을 했다

**스스로 점검**
- [ ] 어텐션 가중치와 다음 토큰 확률이 "같은 softmax"인 이유를 안다
- [ ] T로 나누면 왜 분포가 평평해지는지 손계산으로 보일 수 있다
- [ ] 프롬프트 기본기 3종(명확한 지시·few-shot·형식 지정)을 안다
- [ ] 환각이 구조적인 이유("찾아오기"가 아니라 "생성")를 안다

**🔹심화 (선택)**
- **few-shot 실험:** 분류 프롬프트에 예시 2개(리뷰→라벨)를 붙이고, 애매한 리뷰("그냥 평범해요")의 답이 어떻게 달라지는지 비교하세요(키 필요).
- **3분류:** 라벨을 긍정/중립/부정으로 늘려 보세요 — 애매한 리뷰가 '중립'으로 가는 경향.
- **provider='openai':** 같은 코드로 GPT 호출 — "프롬프트→텍스트" 흐름이 같음을 확인.
- **샘플링 체험:** greedy 루프의 argmax를 `torch.multinomial(probs, 1)`로 바꾸고 T를 걸어 보세요 — 실행마다 다른 문장(비결정성의 정체).